In [1]:
import glob
import os
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

train = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv")
submission = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv")


/tmp/ipykernel_16/2008091755.py:11: DtypeWarning: Columns (38,39) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv")


In [2]:
import glob
import os
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error

In [3]:
TARGET = "TargetValue"
ID_COLUMN = "TransactionID"

test_ids = test[ID_COLUMN].copy()

all_features = pd.concat([train.drop(columns=[TARGET]), test], ignore_index=True)

date = pd.to_datetime(all_features["TransactionDate"], errors="coerce")

all_features["TransactionYear"] = date.dt.year
all_features["TransactionMonth"] = date.dt.month

all_features["ManufactureYear"] = all_features["ManufactureYear"].where(
    all_features["ManufactureYear"].between(1900, 2020)
)

all_features["AssetAge"] = all_features["TransactionYear"] - all_features["ManufactureYear"]
all_features["AssetAge"] = all_features["AssetAge"].where(all_features["AssetAge"].between(0, 100))

all_features["LogOperationalHours"] = np.log1p(
    all_features["OperationalHoursMeter"].clip(lower=0)
)

for col in ["ProductConfigID", "Spec_FullDescriptor"]:
    all_features[f"{col}_Frequency"] = all_features[col].map(
        all_features[col].value_counts(dropna=False)
    ).astype("float32")

all_features = all_features.drop(
    columns=["AssetID", ID_COLUMN, "TransactionDate"],
    errors="ignore"
)

categorical_columns = all_features.select_dtypes(include="object").columns.tolist()

if "ProductConfigID" in all_features.columns:
    categorical_columns.append("ProductConfigID")

categorical_columns = list(dict.fromkeys(categorical_columns))

for col in categorical_columns:
    all_features[col] = all_features[col].fillna("__MISSING__").astype("category")

X = all_features.iloc[:len(train)].copy()
X_test = all_features.iloc[len(train):].copy()
y = np.log1p(train[TARGET])

bins = pd.qcut(y, q=20, labels=False, duplicates="drop")

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=bins
)

params = {
    "objective": "regression",
    "metric": "rmse",
    "n_estimators": 7000,
    "learning_rate": 0.03,
    "num_leaves": 63,
    "min_child_samples": 40,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "cat_smooth": 20,
    "cat_l2": 10,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
    "force_col_wise": True
}

model = lgb.LGBMRegressor(**params)

model.fit(
    X_train,
    y_train,
    categorical_feature=categorical_columns,
    eval_set=[(X_valid, y_valid)],
    eval_metric="rmse",
    callbacks=[
        lgb.early_stopping(150, verbose=True),
        lgb.log_evaluation(100)
    ]
)

valid_pred = np.clip(
    np.expm1(model.predict(X_valid, num_iteration=model.best_iteration_)),
    0,
    None
)

rmsle = np.sqrt(mean_squared_log_error(np.expm1(y_valid), valid_pred))

print("Best iteration:", model.best_iteration_)
print("Local validation RMSLE:", rmsle)

params["n_estimators"] = max(100, int(model.best_iteration_ * 1.05))

final_model = lgb.LGBMRegressor(**params)
final_model.fit(X, y, categorical_feature=categorical_columns)

submission[ID_COLUMN] = test_ids
submission[TARGET] = np.clip(np.expm1(final_model.predict(X_test)), 0, None)

submission.to_csv("submission.csv", index=False)

print("Created submission.csv")
print(submission.head())

Training until validation scores don't improve for 150 rounds
[100]	valid_0's rmse: 0.248772
[200]	valid_0's rmse: 0.224355
[300]	valid_0's rmse: 0.21751
[400]	valid_0's rmse: 0.214266
[500]	valid_0's rmse: 0.212432
[600]	valid_0's rmse: 0.211213
[700]	valid_0's rmse: 0.210348
[800]	valid_0's rmse: 0.209761
[900]	valid_0's rmse: 0.209305
[1000]	valid_0's rmse: 0.208878
[1100]	valid_0's rmse: 0.208631
[1200]	valid_0's rmse: 0.208357
[1300]	valid_0's rmse: 0.208154
[1400]	valid_0's rmse: 0.207991
[1500]	valid_0's rmse: 0.207816
[1600]	valid_0's rmse: 0.207674
[1700]	valid_0's rmse: 0.207618
[1800]	valid_0's rmse: 0.20751
[1900]	valid_0's rmse: 0.207399
[2000]	valid_0's rmse: 0.20731
[2100]	valid_0's rmse: 0.207233
[2200]	valid_0's rmse: 0.207144
[2300]	valid_0's rmse: 0.207027
[2400]	valid_0's rmse: 0.207026
[2500]	valid_0's rmse: 0.206992
[2600]	valid_0's rmse: 0.206925
[2700]	valid_0's rmse: 0.206909
[2800]	valid_0's rmse: 0.206847
[2900]	valid_0's rmse: 0.206827
[3000]	valid_0's rmse:

# MILESTONE-1

In [4]:
print(train.shape,test.shape)

(138701, 50) (15000, 49)


In [5]:
set(train.columns)-set(test.columns)

{'TargetValue'}

In [6]:
cols=train.select_dtypes(include='number').columns
len(cols)


6

In [7]:
missing=train.isna().sum()
missing[missing > 124800]

col4     128320
col5     128320
col18    138635
col19    138635
dtype: int64

In [8]:
train['OperationalHoursMeter'].isna().mean()*100

np.float64(40.838205924975306)

In [9]:
train['TargetValue'].median()

35000.0

In [10]:
train['ManufactureYear'].mode()[0]

np.int64(1001)

In [11]:
train['TransactionDate']=pd.to_datetime(train['TransactionDate'])
train['TransactionDate'].min()

Timestamp('1990-01-31 00:00:00')

In [12]:
train['Spec_BaseClass'].nunique()

1249

In [13]:
train['RegionCode'].value_counts().head()

RegionCode
Florida       23460
Texas         18284
California     9525
Washington     6063
Georgia        5240
Name: count, dtype: int64

In [14]:
train[train['RegionCode']=='Florida']['TargetValue'].mean()

np.float64(45007.54262574595)

In [15]:
train['UtilizationTier'].value_counts(dropna=False)

UtilizationTier
NaN       88226
Medium    24588
Low       17005
High       8882
Name: count, dtype: int64

In [16]:
train[['TargetValue','OperationalHoursMeter']].corr()

,TargetValue,OperationalHoursMeter
TargetValue,1.000000,0.001994
OperationalHoursMeter,0.001994,1.000000


In [17]:
train['FunctionalClassification'].value_counts().head()

FunctionalClassification
Hydraulic Excavator, Track - 21.0 to 24.0 Metric Tons      11158
Motorgrader - 145.0 to 170.0 Horsepower                     8107
Hydraulic Excavator, Track - 12.0 to 14.0 Metric Tons       7625
Backhoe Loader - 14.0 to 15.0 Ft Standard Digging Depth     7603
Hydraulic Excavator, Track - 33.0 to 40.0 Metric Tons       7440
Name: count, dtype: int64